# Feature engineering

In [ ]:
import pandas as pd

daily_sales = pd.read_csv(
    "../data/processed/daily_sales.csv",
    parse_dates=["Date"]
)

print(daily_sales.head())
print(daily_sales.dtypes)

### **Feature 1 - Time features**

In [ ]:
daily_sales["Year"] = daily_sales["Date"].dt.year
daily_sales["Month"] = daily_sales["Date"].dt.month
daily_sales["DayOfWeek"] = daily_sales["Date"].dt.dayofweek

### **Feature 2 - Weekend indicator**

In [ ]:
daily_sales["IsWeekend"] = (
    daily_sales["DayOfWeek"] >= 5
).astype(int)

### **Lag Features**

In [ ]:
daily_sales["Lag_1"] = (
    daily_sales["Revenue"].shift(1)
)

### **Previous 7 day revenue**

In [ ]:
daily_sales["Lag_7"] = (
    daily_sales["Revenue"].shift(7)
)

### **Previous 14 day revenue**

In [ ]:
daily_sales["Lag_14"] = (
    daily_sales["Revenue"].shift(14)
)

### **Rolling features**

In [ ]:
daily_sales["RollingMean_7"] = (
    daily_sales["Revenue"]
    .shift(1)
    .rolling(7)
    .mean()
)

daily_sales["RollingMean_14"] = (
    daily_sales["Revenue"]
    .shift(1)
    .rolling(14)
    .mean()
)

daily_sales["RollingMean_28"] = (
    daily_sales["Revenue"]
    .shift(1)
    .rolling(28)
    .mean()
)

In [ ]:
print(daily_sales.columns.tolist())

### **Forecast dataset**

In [ ]:
forecast_df = daily_sales[
    [
        "Date",
        "Year",
        "Month",
        "DayOfWeek",
        "IsWeekend",
        "Lag_1",
        "Lag_7",
        "Lag_14",
        "RollingMean_7",
        "Revenue"
    ]
].copy()

forecast_df = (
    forecast_df
    .dropna()
    .reset_index(drop=True)
)

forecast_df.head()

In [ ]:
forecast_df = (
    forecast_df
    .dropna()
    .reset_index(drop=True)
)

In [ ]:
print("Forecasting dataset shape:", forecast_df.shape)

print("\nColumns:")
print(forecast_df.columns.tolist())

print("\nMissing values:")
print(forecast_df.isnull().sum())

forecast_df.head()

In [ ]:
forecast_df.to_csv(
    "../data/processed/forecast_dataset.csv",
    index=False
)

### **Prepare X and Y**

In [ ]:
features = [
    "Year",
    "Month",
    "DayOfWeek",
    "IsWeekend",
    "Lag_1",
    "Lag_7",
    "Lag_14",
    "RollingMean_7"
]

X = forecast_df[features]
y = forecast_df["Revenue"]

**Check:**

In [ ]:
print("Features:")
print(X.columns.tolist())

print("\nX shape:", X.shape)
print("y shape:", y.shape)

### **Split the data set**

In [ ]:
n = len(forecast_df)

train_end = int(n * 0.70)
validation_end = int(n * 0.85)

train_df = forecast_df.iloc[:train_end]
validation_df = forecast_df.iloc[train_end:validation_end]
test_df = forecast_df.iloc[validation_end:]

In [ ]:
X_train = train_df[features]
y_train = train_df["Revenue"]

X_validation = validation_df[features]
y_validation = validation_df["Revenue"]

X_test = test_df[features]
y_test = test_df["Revenue"]

**Check:**

In [ ]:
print("Training:", X_train.shape)
print("Validation:", X_validation.shape)
print("Test:", X_test.shape)

print("\nTraining dates:")
print(train_df["Date"].min(), "to", train_df["Date"].max())

print("\nValidation dates:")
print(validation_df["Date"].min(), "to", validation_df["Date"].max())

print("\nTest dates:")
print(test_df["Date"].min(), "to", test_df["Date"].max())

### **Train the first model**

**Linear Regression:**

In [ ]:
from sklearn.linear_model import LinearRegression

linear_model = LinearRegression()

linear_model.fit(
    X_train,
    y_train
)

**Predict validation:**

In [ ]:
linear_validation_pred = linear_model.predict(
    X_validation
)

**Evaluate:**

In [ ]:
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score

mae = mean_absolute_error(
    y_validation,
    linear_validation_pred
)

rmse = np.sqrt(
    mean_squared_error(
        y_validation,
        linear_validation_pred
    )
)

r2 = r2_score(
    y_validation,
    linear_validation_pred
)

print("Linear Regression")
print("MAE :", mae)
print("RMSE:", rmse)
print("R²  :", r2)